# 7 · Host the HalluScan backend on Colab (GPU) + ngrok

Runs the **same** `backend/app.py` as the local demo on a Colab **T4** (single Instruct model ~6 GB + DeBERTa NLI ~0.7 GB), exposed at a **fixed ngrok URL**. The Vercel frontend already points at that URL, so there is **nothing to paste** — just keep this notebook running.

It serves the **Option-B per-sentence detector** automatically (`HALLKING_SENTENCE_TAG=s1`), loading `artifacts/*/*_sentence_s1.*` + `models/fusion_claim_s1.pkl`. A long answer is split into sentences and each factual claim is scored.

**Every session:** fill the two tokens in the CONFIG cell → **Runtime ▸ Run all**. **Keep this tab open** — closing it drops the tunnel.

### 0 · CONFIG — the only cell you edit
Paste your two tokens, then run all. ⚠️ **These tokens are runtime-only — never commit this notebook with them filled in** (the GitHub copy must stay blank). `NGROK_DOMAIN` is the reserved static domain so the URL never changes; leave it as-is.

In [ ]:
# ===== CONFIG (edit the two tokens, then Runtime > Run all) ==============================
REPO            = 'https://github.com/dinitha2004/HalluScan.git'
HF_TOKEN        = ''   # huggingface.co/settings/tokens (Read). Llama-3.1 license must be accepted.
NGROK_AUTHTOKEN = ''   # dashboard.ngrok.com -> Your Authtoken
NGROK_DOMAIN    = 'declared-angular-matchbox.ngrok-free.dev'  # reserved static domain (leave as-is)
# =======================================================================================
assert HF_TOKEN.strip(), 'Paste HF_TOKEN above (huggingface.co/settings/tokens).'
assert NGROK_AUTHTOKEN.strip(), 'Paste NGROK_AUTHTOKEN above (dashboard.ngrok.com -> Your Authtoken).'
print('config set; domain =', NGROK_DOMAIN or '(random)')

### 1 · Get the code + install deps
The repo is **public**, so no token is needed to clone. (`git clone` makes a `HalluScan/` folder.) The ML stack is installed explicitly (Colab already has a compatible torch).

In [ ]:
import os
if not os.path.isdir('HalluScan'):
    !git clone $REPO
%cd HalluScan
!pip install -q transformers accelerate bitsandbytes sentencepiece scikit-learn pandas pyarrow
!pip install -q fastapi 'uvicorn[standard]' pyngrok pysbd nest_asyncio requests

### 2 · Hugging Face login (gated Llama-3.1-8B)
Uses `HF_TOKEN` from the CONFIG cell (must have accepted the Llama-3.1 license). First load downloads the 8B weights ~16 GB — a few minutes on a fresh runtime.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN) if HF_TOKEN.strip() else login()

### 3 · Start the API (background thread) + open the fixed ngrok tunnel
Sets the ngrok authtoken, then opens the tunnel on your **reserved static domain** so the URL is always the same. `nest_asyncio` lets uvicorn run inside Colab's event loop; the model loads on startup (~1–2 min on T4 after weights are cached).

In [ ]:
import sys, threading, uvicorn, nest_asyncio
nest_asyncio.apply()
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN.strip()
sys.path.insert(0, 'backend'); sys.path.insert(0, 'hallking')
import app  # backend/app.py (FastAPI instance = app.app); auto-serves Option B (tag s1)
def _serve():
    uvicorn.run(app.app, host='0.0.0.0', port=8000, log_level='info')
threading.Thread(target=_serve, daemon=True).start()
ngrok.kill()  # drop any tunnel from a previous run before reclaiming the static domain
public = (ngrok.connect(8000, 'http', domain=NGROK_DOMAIN) if NGROK_DOMAIN.strip()
          else ngrok.connect(8000, 'http'))
print('\n========================================================')
print(' PUBLIC URL (the frontend already targets this):')
print('  ', public.public_url)
print('========================================================\n')
# (no ngrok? fallback: !pip install cloudflared && use a cloudflared quick tunnel on port 8000)

### 4 · Wait for model load + smoke-test
Polls `/status` until the model is loaded (check `regime=sentence`, `sentence_tag=s1`, and the calibrated `t_med`/`t_high`), then runs a short and a long question. The long one should split into multiple sentences — confirming the per-sentence pipeline works end to end. After this, open the Vercel site (no pasting needed) and ask away.

In [ ]:
import requests, time
for _ in range(60):
    try:
        s = requests.get('http://localhost:8000/status', timeout=5).json()
        if s.get('model_loaded'):
            print('status:', s); break
    except Exception:
        pass
    time.sleep(5)
else:
    print('model still loading — re-run this cell in a moment')
for q in ['Who painted the Mona Lisa?', 'Tell me about the Sigiriya rock fortress.']:
    out = requests.post('http://localhost:8000/infer', json={'question': q}, timeout=120).json()
    agg = out['aggregate']
    print(f"\nQ: {q}\nA: {out['answer'][:160]}")
    print(f"  -> {agg['label']} (fused={agg['fused']}) | {agg['n_flagged']}/{agg['n_sentences']} flagged")